# 01 · Data Exploration

This notebook explores the TCR–peptide binding datasets used in this project.

**Goals:**
- Understand the distribution of sequence lengths
- Examine class balance and amino acid composition
- Identify potential data-quality issues before modelling

## 1.1 Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

from src.data import create_sample_data, download_vdjdb, generate_negatives

# Reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120})
print("Dependencies loaded ✓")

## 1.2 Load data

We use synthetic data here so the notebook runs offline without any downloads.
Swap `create_sample_data()` for `download_vdjdb()` to use the real VDJdb database.

In [ ]:
# --- Synthetic data (offline demo) ---
df_raw = create_sample_data(n_samples=2000, seed=42)

# To use real VDJdb data instead, uncomment:
# positives = download_vdjdb(save_path='../data/raw/vdjdb.csv')
# df_raw    = generate_negatives(positives, ratio=1.0)

print(df_raw.shape)
df_raw.head()

## 1.3 Class distribution

In [ ]:
counts = df_raw["label"].value_counts()
print("Label counts:")
print(counts.to_string())
print(f"\nPositive rate: {counts[1] / len(df_raw):.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Non-binding (0)", "Binding (1)"], counts.values,
       color=["#d62728", "#1f77b4"], alpha=0.85, edgecolor="white")
ax.set_ylabel("Count")
ax.set_title("Class Distribution", fontweight="bold")
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 1.4 Sequence length distributions

In [ ]:
df_raw["tcr_len"]     = df_raw["tcr_sequence"].str.len()
df_raw["peptide_len"] = df_raw["peptide_sequence"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title, colour in zip(
    axes,
    ["tcr_len", "peptide_len"],
    ["TCR CDR3 Length", "Peptide Length"],
    ["#1f77b4", "#ff7f0e"],
):
    sns.histplot(data=df_raw, x=col, hue="label", bins=15,
                 multiple="dodge", ax=ax, palette={0: "#d62728", 1: colour})
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Sequence length (aa)")

plt.suptitle("Sequence Length Distributions by Binding Label", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(df_raw[["tcr_len", "peptide_len"]].describe().round(1))

## 1.5 Amino acid composition

In [ ]:
def aa_freq(sequences, label=""):
    counts = Counter("".join(sequences))
    total  = sum(counts.values())
    return {aa: counts.get(aa, 0) / total for aa in "ACDEFGHIKLMNPQRSTVWY"}

tcr_freq_pos = aa_freq(df_raw.loc[df_raw.label == 1, "tcr_sequence"], "Binders")
tcr_freq_neg = aa_freq(df_raw.loc[df_raw.label == 0, "tcr_sequence"], "Non-binders")

freq_df = pd.DataFrame({
    "Binders":     tcr_freq_pos,
    "Non-binders": tcr_freq_neg,
}).sort_index()

fig, ax = plt.subplots(figsize=(13, 4))
freq_df.plot(kind="bar", ax=ax, alpha=0.85, edgecolor="white", width=0.7)
ax.set_xlabel("Amino acid")
ax.set_ylabel("Relative frequency")
ax.set_title("Amino Acid Composition in TCR CDR3 Sequences", fontweight="bold")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 1.6 Summary statistics

In [ ]:
print("=" * 55)
print("  DATASET SUMMARY")
print("=" * 55)
print(f"  Total pairs      : {len(df_raw):,}")
print(f"  Positive (bind.) : {(df_raw.label == 1).sum():,}")
print(f"  Negative         : {(df_raw.label == 0).sum():,}")
print(f"  TCR length range : {df_raw.tcr_len.min()}–{df_raw.tcr_len.max()} aa")
print(f"  Pep length range : {df_raw.peptide_len.min()}–{df_raw.peptide_len.max()} aa")
print("=" * 55)
print("\nProceed to notebook 02 to generate embeddings and graphs ►")